# 05. Small TimeMixer++ 계열 학습

60개 1분봉 OHLC 파생 sequence로 **3분 안에 +3% 도달 가능성**을 학습한다. 날짜 기반 expanding walk-forward OOF에서 epoch를 선택한 뒤, 7개 Train 날짜 전체로 다시 학습하고 고정 Test 2일을 한 번 평가한다.

이번 노트북은 세 가지 target 정의를 같은 백본과 같은 split에서 분리해 비교한다.

| strategy | target | loss | OOF epoch 선택 기준 |
|---|---|---|---|
| `open_hard` | `t+1 open` 기준 3분 +3% 여부 | 무가중 `BCEWithLogitsLoss` | hard-label PR-AUC 최대 |
| `random_soft` | `t+1 low~high` 균등 진입의 +3% 확률 | soft-target BCE | soft BCE 최소 |
| `mfe_mae` | 3분 MFE와 downside(`-MAE`) | train-q99 clipping + robust scaling + `SmoothL1(beta=0.5)` | 두 target Spearman 평균 최대 |

> **구현 범위:** [TimeMixer++ (ICLR 2025)](https://proceedings.iclr.cc/paper_files/paper/2025/file/2b187165e28fdfdc0ffb34d1bfff2b0c-Paper-Conference.pdf)의 multi-scale/multi-resolution 아이디어와 공개 [TimeMixer 저장소](https://github.com/kwuking/TimeMixer)의 계절 bottom-up / 추세 top-down PDM 흐름을 60봉 분류·회귀에 맞게 축소한 task-specific adaptation이다. 논문의 전체 benchmark 구현을 byte-for-byte 재현한 모델은 아니다.

Test(2026-07-16~17)는 이미 과거 개발에서 본 날짜라 **고정 development holdout**이며 최종 pristine 검증이 아니다.

In [1]:
from pathlib import Path
import json
import math
import random
import time
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model/timemixerpp_ohlc_60m_v1').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/training/timemixerpp_ohlc_60m_v1').resolve()
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_VERSION = 'ohlc_60m_tp3pct_v1'
RANDOM_VERSION = 'ohlc_60m_tp3pct_random_entry_v1'
SEED = 20260721
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'sequence_length': 60,
    'input_features': 11,
    'context_features': 20,
    'd_model': 32,
    'd_ff': 64,
    'num_blocks': 2,
    'moving_average': 5,
    'dropout': 0.15,
    'batch_size': 512,
    'candidate_epochs': [4, 8, 12],
    'learning_rate': 3e-4,
    'weight_decay': 1e-3,
    'gradient_clip': 1.0,
    'train_stride_minutes': 3,
    'input_clip': 10.0,
    'regression_quantile_clip': 0.99,
    'smooth_l1_beta': 0.5,
    'num_workers': 0,
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
torch.set_float32_matmul_precision('high')

print('project:', PROJECT_ROOT)
print('device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)
print('model output:', MODEL_ROOT)
print('result output:', RESULT_ROOT)

project: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
device: cuda
GPU: NVIDIA RTX 6000 Ada Generation
torch: 2.9.1+cu128
model output: /home/user/urbandatalab/YSLee/model/timemixerpp_ohlc_60m_v1
result output: /home/user/urbandatalab/YSLee/results/training/timemixerpp_ohlc_60m_v1


## 1. 전처리 artifact와 날짜 split 로드

NPZ 배열 순서와 metadata / random-label / split의 `sample_id`가 완전히 같은지 검사한다. scaling은 저장 데이터에 미리 적용하지 않고 각 OOF fit 날짜에서만 계산한다.

In [2]:
schema = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_schema.json').read_text(encoding='utf-8'))
fold_document = json.loads((PREPROCESS_ROOT / f'{DATA_VERSION}_walk_forward_folds.json').read_text(encoding='utf-8'))

metadata = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_metadata.parquet')
random_labels = pd.read_parquet(PREPROCESS_ROOT / f'{RANDOM_VERSION}_metadata.parquet')
split = pd.read_parquet(PREPROCESS_ROOT / f'{DATA_VERSION}_split.parquet')

with np.load(PREPROCESS_ROOT / f'{DATA_VERSION}_features.npz') as arrays:
    sequence = arrays['sequence'].astype(np.float32, copy=False)
    context = arrays['context'].astype(np.float32, copy=False)

assert metadata['sample_id'].is_unique
assert metadata['sample_id'].tolist() == random_labels['sample_id'].tolist() == split['sample_id'].tolist()
assert len(metadata) == len(sequence) == len(context)
assert sequence.shape[1:] == (CONFIG['sequence_length'], CONFIG['input_features'])
assert context.shape[1] == CONFIG['context_features']
assert np.isfinite(sequence).all() and np.isfinite(context).all()
assert fold_document['test_is_pristine'] is False

y_hard = metadata['target_tp3_3m'].to_numpy(np.float32)
y_soft = random_labels['p_tp3_random_entry_3m'].to_numpy(np.float32)
y_reg = np.column_stack([
    metadata['mfe_3m'].to_numpy(np.float32),
    -metadata['mae_3m'].to_numpy(np.float32),
]).astype(np.float32)

assert np.isin(y_hard, [0.0, 1.0]).all()
assert ((0 <= y_soft) & (y_soft <= 1)).all()
assert (y_reg >= -1e-7).all()

# 한 run 안에서 매분 생성된 60봉 window의 중복을 줄인다. OOF/Test 평가는 모든 시점에서 한다.
within_run_position = metadata.groupby('run_id', sort=False).cumcount().to_numpy()
train_sampling_mask = (within_run_position % CONFIG['train_stride_minutes']) == 0

train_mask = split['split'].eq('train').to_numpy()
test_mask = split['split'].eq('test').to_numpy()
oof_mask = split['oof_fold'].gt(0).to_numpy()

dataset_summary = pd.DataFrame({
    'partition': ['Train 전체', 'Train 학습 stride=3', 'OOF 평가', 'Test 평가'],
    'samples': [
        int(train_mask.sum()),
        int((train_mask & train_sampling_mask).sum()),
        int(oof_mask.sum()),
        int(test_mask.sum()),
    ],
})
display(dataset_summary)
display(pd.DataFrame({
    'target': ['open hard TP3/3m', 'random-entry soft TP3/3m'],
    'Train mean': [float(y_hard[train_mask].mean()), float(y_soft[train_mask].mean())],
    'Test mean': [float(y_hard[test_mask].mean()), float(y_soft[test_mask].mean())],
}).style.format({'Train mean': '{:.3%}', 'Test mean': '{:.3%}'}))
print('sequence:', sequence.shape, 'context:', context.shape)
print('OOF folds:', len(fold_document['folds']))

,partition,samples
0,Train 전체,53047
1,Train 학습 stride=3,17756
2,OOF 평가,31081
3,Test 평가,11097


,target,Train mean,Test mean
0,open hard TP3/3m,12.370%,9.525%
1,random-entry soft TP3/3m,11.660%,8.631%


sequence: (64144, 60, 11) context: (64144, 20)
OOF folds: 4


## 2. Small TimeMixer++ 계열 백본

해상도 60→30→15에서 moving-average로 계절/추세를 분해한다. 계절 성분은 고해상도에서 저해상도로 bottom-up mixing하고, 추세 성분은 저해상도에서 고해상도로 top-down mixing한다. 각 해상도의 mean/max/last pooling과 20개 context encoder를 합쳐 target head로 보낸다.

OHLC의 절대 가격은 입력하지 않으며 전처리에서 만든 상대 수익률·캔들 형태만 사용한다.

In [3]:
class MovingAverageDecomposition(nn.Module):
    def __init__(self, kernel_size=5):
        super().__init__()
        self.kernel_size = kernel_size

    def forward(self, x):
        # x: [B, T, D]
        left = (self.kernel_size - 1) // 2
        right = self.kernel_size - 1 - left
        padded = F.pad(x.transpose(1, 2), (left, right), mode='replicate')
        trend = F.avg_pool1d(padded, kernel_size=self.kernel_size, stride=1).transpose(1, 2)
        season = x - trend
        return season, trend


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MultiResolutionPDMBlock(nn.Module):
    def __init__(self, lengths, d_model, d_ff, moving_average, dropout):
        super().__init__()
        self.lengths = lengths
        self.decomposition = MovingAverageDecomposition(moving_average)
        self.season_ff = nn.ModuleList([FeedForward(d_model, d_ff, dropout) for _ in lengths])
        self.trend_ff = nn.ModuleList([FeedForward(d_model, d_ff, dropout) for _ in lengths])
        self.season_down = nn.ModuleList([
            nn.Sequential(nn.Linear(lengths[i], lengths[i + 1]), nn.GELU())
            for i in range(len(lengths) - 1)
        ])
        self.trend_up = nn.ModuleList([
            nn.Sequential(nn.Linear(lengths[i + 1], lengths[i]), nn.GELU())
            for i in range(len(lengths) - 1)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in lengths])
        self.output_ff = nn.ModuleList([FeedForward(d_model, d_ff, dropout) for _ in lengths])

    def forward(self, resolutions):
        seasons, trends = [], []
        for i, x in enumerate(resolutions):
            season, trend = self.decomposition(x)
            seasons.append(self.season_ff[i](season))
            trends.append(self.trend_ff[i](trend))

        # high resolution -> low resolution seasonal mixing
        mixed_seasons = [seasons[0]]
        for i in range(len(seasons) - 1):
            down = self.season_down[i](mixed_seasons[-1].transpose(1, 2)).transpose(1, 2)
            mixed_seasons.append(seasons[i + 1] + down)

        # low resolution -> high resolution trend mixing
        mixed_trends = list(trends)
        for i in reversed(range(len(trends) - 1)):
            up = self.trend_up[i](mixed_trends[i + 1].transpose(1, 2)).transpose(1, 2)
            mixed_trends[i] = trends[i] + up

        outputs = []
        for i, original in enumerate(resolutions):
            mixed = mixed_seasons[i] + mixed_trends[i]
            outputs.append(original + self.output_ff[i](self.norms[i](mixed)))
        return outputs


class SmallTimeMixerPP(nn.Module):
    def __init__(
        self,
        input_features,
        context_features,
        output_dim,
        d_model=32,
        d_ff=64,
        num_blocks=2,
        moving_average=5,
        dropout=0.15,
    ):
        super().__init__()
        self.lengths = [60, 30, 15]
        self.input_projection = nn.Linear(input_features, d_model)
        self.position_embeddings = nn.ParameterList([
            nn.Parameter(torch.zeros(1, length, d_model)) for length in self.lengths
        ])
        self.blocks = nn.ModuleList([
            MultiResolutionPDMBlock(
                self.lengths, d_model, d_ff, moving_average, dropout
            )
            for _ in range(num_blocks)
        ])
        self.final_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in self.lengths])
        self.context_encoder = nn.Sequential(
            nn.Linear(context_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        fused_dim = len(self.lengths) * d_model * 3 + d_model
        self.head = nn.Sequential(
            nn.Linear(fused_dim, d_ff * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff * 2, output_dim),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, sequence_x, context_x):
        finest = self.input_projection(sequence_x)
        resolutions = [finest]
        for _ in range(2):
            resolutions.append(F.avg_pool1d(
                resolutions[-1].transpose(1, 2), kernel_size=2, stride=2
            ).transpose(1, 2))
        resolutions = [x + pos for x, pos in zip(resolutions, self.position_embeddings)]

        for block in self.blocks:
            resolutions = block(resolutions)

        pooled = []
        for x, norm in zip(resolutions, self.final_norms):
            x = norm(x)
            pooled.extend([x.mean(dim=1), x.amax(dim=1), x[:, -1]])
        pooled.append(self.context_encoder(context_x))
        return self.head(torch.cat(pooled, dim=1))


smoke_model = SmallTimeMixerPP(
    input_features=CONFIG['input_features'],
    context_features=CONFIG['context_features'],
    output_dim=1,
    d_model=CONFIG['d_model'],
    d_ff=CONFIG['d_ff'],
    num_blocks=CONFIG['num_blocks'],
    moving_average=CONFIG['moving_average'],
    dropout=CONFIG['dropout'],
).to(DEVICE)
with torch.inference_mode():
    smoke_output = smoke_model(
        torch.zeros(2, 60, 11, device=DEVICE),
        torch.zeros(2, 20, device=DEVICE),
    )
print('parameters:', f'{sum(p.numel() for p in smoke_model.parameters()):,}')
print('smoke output:', tuple(smoke_output.shape))
del smoke_model, smoke_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

parameters: 130,999
smoke output: (2, 1)


## 3. Fold-local scaling, loss, metric

- 입력 median/IQR은 각 fold의 실제 학습 subset에서만 계산하고 ±10 IQR로 clip한다.
- 회귀 target도 fold Train의 q99와 median/IQR만 사용한다.
- BCE에는 class weight와 focal loss를 넣지 않는다. 먼저 확률 calibration을 보존한 기준 성능을 확인한다.
- threshold/체결 조건/백테스트는 이 학습 노트북의 범위가 아니다.

In [4]:
STRATEGIES = {
    'open_hard': {'output_dim': 1, 'selection_metric': 'hard_pr_auc', 'mode': 'max'},
    'random_soft': {'output_dim': 1, 'selection_metric': 'soft_bce', 'mode': 'min'},
    'mfe_mae': {'output_dim': 2, 'selection_metric': 'mean_target_spearman', 'mode': 'max'},
}


def robust_center_scale(values, axis=0):
    center = np.median(values, axis=axis).astype(np.float32)
    q25 = np.quantile(values, 0.25, axis=axis).astype(np.float32)
    q75 = np.quantile(values, 0.75, axis=axis).astype(np.float32)
    scale = (q75 - q25).astype(np.float32)
    scale = np.where(scale < 1e-6, 1.0, scale).astype(np.float32)
    return center, scale


def fit_input_scaler(indices):
    seq_values = sequence[indices].reshape(-1, sequence.shape[-1])
    seq_center, seq_scale = robust_center_scale(seq_values, axis=0)
    ctx_center, ctx_scale = robust_center_scale(context[indices], axis=0)
    return {
        'sequence_center': seq_center,
        'sequence_scale': seq_scale,
        'context_center': ctx_center,
        'context_scale': ctx_scale,
    }


def transform_inputs(indices, scaler):
    seq = (sequence[indices] - scaler['sequence_center'][None, None, :]) / scaler['sequence_scale'][None, None, :]
    ctx = (context[indices] - scaler['context_center'][None, :]) / scaler['context_scale'][None, :]
    seq = np.clip(seq, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    ctx = np.clip(ctx, -CONFIG['input_clip'], CONFIG['input_clip']).astype(np.float32)
    return np.ascontiguousarray(seq), np.ascontiguousarray(ctx)


def fit_regression_scaler(indices):
    values = y_reg[indices]
    upper = np.quantile(values, CONFIG['regression_quantile_clip'], axis=0).astype(np.float32)
    clipped = np.minimum(values, upper)
    center, scale = robust_center_scale(clipped, axis=0)
    return {'upper': upper, 'center': center, 'scale': scale}


def transform_regression_target(indices, scaler):
    values = np.minimum(y_reg[indices], scaler['upper'])
    return ((values - scaler['center']) / scaler['scale']).astype(np.float32)


def inverse_regression_target(values, scaler):
    restored = values * scaler['scale'][None, :] + scaler['center'][None, :]
    return np.clip(restored, 0.0, scaler['upper'][None, :]).astype(np.float32)


def make_loader(indices, scaler, strategy, target_scaler=None, shuffle=False, seed=SEED):
    seq, ctx = transform_inputs(indices, scaler)
    if strategy == 'open_hard':
        target = y_hard[indices, None]
    elif strategy == 'random_soft':
        target = y_soft[indices, None]
    else:
        target = transform_regression_target(indices, target_scaler)
    dataset = TensorDataset(
        torch.from_numpy(seq),
        torch.from_numpy(ctx),
        torch.from_numpy(np.ascontiguousarray(target, dtype=np.float32)),
    )
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=shuffle,
        num_workers=CONFIG['num_workers'],
        pin_memory=DEVICE.type == 'cuda',
        drop_last=False,
        generator=generator,
    )


def build_model(strategy):
    return SmallTimeMixerPP(
        input_features=CONFIG['input_features'],
        context_features=CONFIG['context_features'],
        output_dim=STRATEGIES[strategy]['output_dim'],
        d_model=CONFIG['d_model'],
        d_ff=CONFIG['d_ff'],
        num_blocks=CONFIG['num_blocks'],
        moving_average=CONFIG['moving_average'],
        dropout=CONFIG['dropout'],
    ).to(DEVICE)


def loss_for(strategy, output, target):
    if strategy in ('open_hard', 'random_soft'):
        return F.binary_cross_entropy_with_logits(output, target)
    return F.smooth_l1_loss(output, target, beta=CONFIG['smooth_l1_beta'])


def predict(model, loader):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for seq_batch, ctx_batch, _ in loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
            outputs.append(output.float().cpu().numpy())
    return np.concatenate(outputs)


def clipped_probability(logits):
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    return np.clip(probabilities, 1e-7, 1 - 1e-7)


def soft_binary_cross_entropy(target, probability):
    return float(-np.mean(target * np.log(probability) + (1 - target) * np.log(1 - probability)))


def safe_spearman(actual, predicted):
    value = spearmanr(actual, predicted).statistic
    return float(value) if np.isfinite(value) else np.nan


def classification_metrics(indices, logits, strategy):
    probability = clipped_probability(logits.reshape(-1))
    hard = y_hard[indices]
    soft = y_soft[indices]
    return {
        'hard_pr_auc': float(average_precision_score(hard, probability)),
        'hard_roc_auc': float(roc_auc_score(hard, probability)) if np.unique(hard).size > 1 else np.nan,
        'hard_log_loss': float(log_loss(hard, probability, labels=[0, 1])),
        'hard_brier': float(brier_score_loss(hard, probability)),
        'soft_bce': soft_binary_cross_entropy(soft, probability),
        'soft_brier': float(np.mean((soft - probability) ** 2)),
        'soft_spearman': safe_spearman(soft, probability),
        'prediction_mean': float(probability.mean()),
        'hard_prevalence': float(hard.mean()),
        'soft_target_mean': float(soft.mean()),
    }


def regression_metrics(indices, scaled_prediction, target_scaler):
    prediction = inverse_regression_target(scaled_prediction, target_scaler)
    actual = y_reg[indices]
    score_pred = prediction[:, 0] - prediction[:, 1]
    score_actual = actual[:, 0] - actual[:, 1]
    return {
        'mfe_spearman': safe_spearman(actual[:, 0], prediction[:, 0]),
        'downside_spearman': safe_spearman(actual[:, 1], prediction[:, 1]),
        'mean_target_spearman': float(np.nanmean([
            safe_spearman(actual[:, 0], prediction[:, 0]),
            safe_spearman(actual[:, 1], prediction[:, 1]),
        ])),
        'mfe_mae': float(mean_absolute_error(actual[:, 0], prediction[:, 0])),
        'downside_mae': float(mean_absolute_error(actual[:, 1], prediction[:, 1])),
        'utility_spearman': safe_spearman(score_actual, score_pred),
        'utility_hard_pr_auc': float(average_precision_score(y_hard[indices], score_pred)),
        'predicted_mfe_mean': float(prediction[:, 0].mean()),
        'predicted_downside_mean': float(prediction[:, 1].mean()),
    }


def evaluate_predictions(indices, raw_prediction, strategy, target_scaler=None):
    if strategy in ('open_hard', 'random_soft'):
        return classification_metrics(indices, raw_prediction, strategy)
    return regression_metrics(indices, raw_prediction, target_scaler)


def train_epochs(strategy, fit_indices, eval_indices, candidate_epochs, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_regression_scaler(fit_indices) if strategy == 'mfe_mae' else None
    train_loader = make_loader(
        fit_indices, input_scaler, strategy, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, strategy, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model(strategy)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(candidate_epochs), eta_min=CONFIG['learning_rate'] * 0.1
    )
    history = []
    snapshots = {}
    predictions = {}

    for epoch in range(1, max(candidate_epochs) + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = loss_for(strategy, output, target_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'epoch': epoch,
            'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if epoch in candidate_epochs:
            raw_prediction = predict(model, eval_loader)
            predictions[epoch] = raw_prediction
            snapshots[epoch] = deepcopy(model.state_dict())

    del train_loader, eval_loader, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'predictions': predictions,
        'snapshots': snapshots,
        'history': history,
        'input_scaler': input_scaler,
        'target_scaler': target_scaler,
    }

## 4. Expanding walk-forward OOF 학습

Fold마다 과거 날짜만 fit하고 바로 다음 한 날짜 전체를 예측한다. 후보 epoch 4/8/12의 OOF 예측을 모두 보관하며, Test는 이 셀에서 읽거나 평가하지 않는다.

In [5]:
fold_records = []
history_records = []
oof_raw_predictions = {
    strategy: {epoch: np.full((len(metadata), spec['output_dim']), np.nan, dtype=np.float32)
               for epoch in CONFIG['candidate_epochs']}
    for strategy, spec in STRATEGIES.items()
}
oof_target_scalers = {strategy: {} for strategy in STRATEGIES}

oof_start = time.time()
for strategy_index, strategy in enumerate(STRATEGIES):
    print(f'\n===== OOF strategy: {strategy} =====')
    for fold in fold_document['folds']:
        fold_number = int(fold['fold'])
        fit_all = np.flatnonzero(split['session'].isin(fold['fit_sessions']).to_numpy())
        fit_indices = fit_all[train_sampling_mask[fit_all]]
        eval_indices = np.flatnonzero(split['session'].eq(fold['evaluation_session']).to_numpy())
        assert split.iloc[fit_indices]['decision_timestamp'].max() < split.iloc[eval_indices]['decision_timestamp'].min()

        run_seed = SEED + strategy_index * 100 + fold_number
        result = train_epochs(
            strategy,
            fit_indices,
            eval_indices,
            CONFIG['candidate_epochs'],
            run_seed,
        )
        oof_target_scalers[strategy][fold_number] = result['target_scaler']

        for row in result['history']:
            history_records.append({
                'phase': 'oof', 'strategy': strategy, 'fold': fold_number,
                'fit_sessions': ','.join(fold['fit_sessions']),
                'evaluation_session': fold['evaluation_session'],
                'fit_samples_after_stride': len(fit_indices), **row,
            })

        for epoch, raw_prediction in result['predictions'].items():
            oof_raw_predictions[strategy][epoch][eval_indices] = raw_prediction
            metrics = evaluate_predictions(
                eval_indices, raw_prediction, strategy, result['target_scaler']
            )
            fold_records.append({
                'strategy': strategy,
                'fold': fold_number,
                'evaluation_session': fold['evaluation_session'],
                'epoch': epoch,
                'fit_samples_after_stride': len(fit_indices),
                'evaluation_samples': len(eval_indices),
                **metrics,
            })
        primary = STRATEGIES[strategy]['selection_metric']
        last_value = fold_records[-1][primary]
        print(
            f"fold {fold_number} | fit {len(fit_indices):,} | eval {len(eval_indices):,} "
            f"| epoch {max(CONFIG['candidate_epochs'])} {primary}={last_value:.5f}"
        )

print(f'\nOOF elapsed: {(time.time() - oof_start) / 60:.2f} min')
fold_metrics = pd.DataFrame(fold_records)
training_history = pd.DataFrame(history_records)
display(fold_metrics.head())


===== OOF strategy: open_hard =====


fold 1 | fit 7,355 | eval 9,077 | epoch 12 hard_pr_auc=0.41835


fold 2 | fit 10,397 | eval 7,297 | epoch 12 hard_pr_auc=0.40502


fold 3 | fit 12,838 | eval 6,120 | epoch 12 hard_pr_auc=0.34912


fold 4 | fit 14,883 | eval 8,587 | epoch 12 hard_pr_auc=0.36248

===== OOF strategy: random_soft =====


...

fold 1 | fit 7,355 | eval 9,077 | epoch 12 mean_target_spearman=0.54537


fold 2 | fit 10,397 | eval 7,297 | epoch 12 mean_target_spearman=0.46538


fold 3 | fit 12,838 | eval 6,120 | epoch 12 mean_target_spearman=0.47814


fold 4 | fit 14,883 | eval 8,587 | epoch 12 mean_target_spearman=0.48138

OOF elapsed: 1.14 min


,strategy,fold,evaluation_session,epoch,fit_samples_after_stride,evaluation_samples,hard_pr_auc,hard_roc_auc,hard_log_loss,hard_brier,...,soft_target_mean,mfe_spearman,downside_spearman,mean_target_spearman,mfe_mae,downside_mae,utility_spearman,utility_hard_pr_auc,predicted_mfe_mean,predicted_downside_mean
0,open_hard,1,session_2026-07-10,4,7355,9077,0.410432,0.802176,0.364588,0.117471,...,0.159144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,open_hard,1,session_2026-07-10,8,7355,9077,0.415923,0.803908,0.361179,0.115967,...,0.159144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,open_hard,1,session_2026-07-10,12,7355,9077,0.418353,0.804443,0.359215,0.114900,...,0.159144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,open_hard,2,session_2026-07-13,4,10397,7297,0.393520,0.807588,0.313601,0.094283,...,0.120917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,open_hard,2,session_2026-07-13,8,10397,7297,0.403399,0.809957,0.311928,0.094373,...,0.120917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. OOF로 epoch 선택

네 OOF 날짜의 예측을 이어 붙여 pooled metric을 계산한다. Test 성능은 아직 계산하지 않는다. pooled metric과 함께 날짜별 최악값도 저장해 날짜 편차를 확인할 수 있게 한다.

In [6]:
epoch_records = []
selected_epochs = {}

oof_indices = np.flatnonzero(oof_mask)
for strategy, spec in STRATEGIES.items():
    for epoch in CONFIG['candidate_epochs']:
        raw = oof_raw_predictions[strategy][epoch][oof_indices]
        assert np.isfinite(raw).all()
        if strategy == 'mfe_mae':
            # 회귀 prediction은 fold별 target scaling이 다르므로 원 단위로 복원해 합친다.
            restored = np.full_like(raw, np.nan)
            for fold in fold_document['folds']:
                fold_number = int(fold['fold'])
                local_mask = split.iloc[oof_indices]['oof_fold'].to_numpy() == fold_number
                restored[local_mask] = inverse_regression_target(
                    raw[local_mask], oof_target_scalers[strategy][fold_number]
                )
            actual = y_reg[oof_indices]
            mfe_s = safe_spearman(actual[:, 0], restored[:, 0])
            down_s = safe_spearman(actual[:, 1], restored[:, 1])
            score_pred = restored[:, 0] - restored[:, 1]
            pooled = {
                'mfe_spearman': mfe_s,
                'downside_spearman': down_s,
                'mean_target_spearman': float(np.nanmean([mfe_s, down_s])),
                'mfe_mae': float(mean_absolute_error(actual[:, 0], restored[:, 0])),
                'downside_mae': float(mean_absolute_error(actual[:, 1], restored[:, 1])),
                'utility_spearman': safe_spearman(actual[:, 0] - actual[:, 1], score_pred),
                'utility_hard_pr_auc': float(average_precision_score(y_hard[oof_indices], score_pred)),
            }
        else:
            pooled = classification_metrics(oof_indices, raw, strategy)

        candidate_fold_rows = fold_metrics[
            fold_metrics['strategy'].eq(strategy) & fold_metrics['epoch'].eq(epoch)
        ]
        primary = spec['selection_metric']
        epoch_records.append({
            'strategy': strategy,
            'epoch': epoch,
            **pooled,
            f'worst_daily_{primary}': (
                float(candidate_fold_rows[primary].min())
                if spec['mode'] == 'max'
                else float(candidate_fold_rows[primary].max())
            ),
        })

epoch_selection = pd.DataFrame(epoch_records)
for strategy, spec in STRATEGIES.items():
    rows = epoch_selection[epoch_selection['strategy'].eq(strategy)]
    metric = spec['selection_metric']
    best_index = rows[metric].idxmax() if spec['mode'] == 'max' else rows[metric].idxmin()
    selected_epochs[strategy] = int(epoch_selection.loc[best_index, 'epoch'])

selection_view_columns = [
    'strategy', 'epoch', 'hard_pr_auc', 'soft_bce',
    'mfe_spearman', 'downside_spearman', 'mean_target_spearman',
]
selection_view_columns = [c for c in selection_view_columns if c in epoch_selection]
display(epoch_selection[selection_view_columns].sort_values(['strategy', 'epoch']))
print('selected epochs:', selected_epochs)

,strategy,epoch,hard_pr_auc,soft_bce,mfe_spearman,downside_spearman,mean_target_spearman
6,mfe_mae,4,NaN,NaN,0.475547,0.492804,0.484176
7,mfe_mae,8,NaN,NaN,0.483966,0.507356,0.495661
8,mfe_mae,12,NaN,NaN,0.485442,0.511218,0.498330
0,open_hard,4,0.376627,0.310259,NaN,NaN,NaN
1,open_hard,8,0.385519,0.306229,NaN,NaN,NaN
2,open_hard,12,0.385697,0.305448,NaN,NaN,NaN
3,random_soft,4,0.387702,0.303423,NaN,NaN,NaN
4,random_soft,8,0.392519,0.301588,NaN,NaN,NaN
5,random_soft,12,0.393518,0.301507,NaN,NaN,NaN


selected epochs: {'open_hard': 12, 'random_soft': 12, 'mfe_mae': 12}


## 6. 전체 Train 재학습과 고정 Test 진단

각 전략은 OOF에서 선택된 epoch까지만 7개 Train 날짜로 새로 학습한다. Test 두 날짜는 여기서 처음 예측하며 model/epoch/threshold 선택에 되돌려 쓰지 않는다.

In [7]:
final_models = {}
final_input_scalers = {}
final_target_scalers = {}
test_raw_predictions = {}
test_metric_records = []

full_train_all = np.flatnonzero(train_mask)
full_train_indices = full_train_all[train_sampling_mask[full_train_all]]
test_indices = np.flatnonzero(test_mask)


def train_final_model(strategy, fit_indices, eval_indices, selected_epoch, run_seed):
    seed_everything(run_seed)
    input_scaler = fit_input_scaler(fit_indices)
    target_scaler = fit_regression_scaler(fit_indices) if strategy == 'mfe_mae' else None
    train_loader = make_loader(
        fit_indices, input_scaler, strategy, target_scaler, shuffle=True, seed=run_seed
    )
    eval_loader = make_loader(
        eval_indices, input_scaler, strategy, target_scaler, shuffle=False, seed=run_seed
    )
    model = build_model(strategy)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(CONFIG['candidate_epochs']),
        eta_min=CONFIG['learning_rate'] * 0.1,
    )
    history = []
    for epoch in range(1, selected_epoch + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for seq_batch, ctx_batch, target_batch in train_loader:
            seq_batch = seq_batch.to(DEVICE, non_blocking=True)
            ctx_batch = ctx_batch.to(DEVICE, non_blocking=True)
            target_batch = target_batch.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.bfloat16,
                enabled=DEVICE.type == 'cuda',
            ):
                output = model(seq_batch, ctx_batch)
                loss = loss_for(strategy, output, target_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
            optimizer.step()
            total_loss += float(loss.detach()) * len(target_batch)
            total_count += len(target_batch)
        scheduler.step()
        history.append({
            'phase': 'final', 'strategy': strategy, 'fold': 0,
            'fit_sessions': ','.join(fold_document['train_sessions']),
            'evaluation_session': ','.join(fold_document['test_sessions']),
            'fit_samples_after_stride': len(fit_indices),
            'epoch': epoch, 'train_loss': total_loss / total_count,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
    raw_prediction = predict(model, eval_loader)
    return model, input_scaler, target_scaler, raw_prediction, history


final_start = time.time()
for strategy_index, strategy in enumerate(STRATEGIES):
    chosen_epoch = selected_epochs[strategy]
    print(f'final {strategy}: {chosen_epoch} epochs')
    model, input_scaler, target_scaler, raw_prediction, history = train_final_model(
        strategy,
        full_train_indices,
        test_indices,
        chosen_epoch,
        SEED + 1000 + strategy_index,
    )
    final_models[strategy] = model
    final_input_scalers[strategy] = input_scaler
    final_target_scalers[strategy] = target_scaler
    test_raw_predictions[strategy] = raw_prediction
    history_records.extend(history)

    overall_metrics = evaluate_predictions(
        test_indices, raw_prediction, strategy, target_scaler
    )
    test_metric_records.append({
        'strategy': strategy, 'scope': 'all_test', 'session': 'ALL',
        'selected_epoch': chosen_epoch, 'samples': len(test_indices), **overall_metrics,
    })
    for session in fold_document['test_sessions']:
        local = np.flatnonzero(split['session'].eq(session).to_numpy())
        positions = np.searchsorted(test_indices, local)
        local_prediction = raw_prediction[positions]
        local_metrics = evaluate_predictions(local, local_prediction, strategy, target_scaler)
        test_metric_records.append({
            'strategy': strategy, 'scope': 'daily', 'session': session,
            'selected_epoch': chosen_epoch, 'samples': len(local), **local_metrics,
        })

print(f'final elapsed: {(time.time() - final_start) / 60:.2f} min')
test_metrics = pd.DataFrame(test_metric_records)
training_history = pd.DataFrame(history_records)

test_view_columns = [
    'strategy', 'scope', 'session', 'samples', 'selected_epoch',
    'hard_pr_auc', 'soft_bce', 'mfe_spearman', 'downside_spearman',
    'mean_target_spearman', 'utility_hard_pr_auc',
]
test_view_columns = [c for c in test_view_columns if c in test_metrics]
display(test_metrics[test_view_columns])

final open_hard: 12 epochs


final random_soft: 12 epochs


final mfe_mae: 12 epochs


final elapsed: 0.42 min


,strategy,scope,session,samples,selected_epoch,hard_pr_auc,soft_bce,mfe_spearman,downside_spearman,mean_target_spearman,utility_hard_pr_auc
0,open_hard,all_test,ALL,11097,12,0.288774,0.236608,NaN,NaN,NaN,NaN
1,open_hard,daily,session_2026-07-16,5338,12,0.289892,0.282028,NaN,NaN,NaN,NaN
2,open_hard,daily,session_2026-07-17,5759,12,0.288798,0.194508,NaN,NaN,NaN,NaN
3,random_soft,all_test,ALL,11097,12,0.293191,0.237546,NaN,NaN,NaN,NaN
4,random_soft,daily,session_2026-07-16,5338,12,0.291732,0.285828,NaN,NaN,NaN,NaN
5,random_soft,daily,session_2026-07-17,5759,12,0.297580,0.192794,NaN,NaN,NaN,NaN
6,mfe_mae,all_test,ALL,11097,12,NaN,NaN,0.474353,0.535550,0.504952,0.125971
7,mfe_mae,daily,session_2026-07-16,5338,12,NaN,NaN,0.480026,0.544963,0.512494,0.137293
8,mfe_mae,daily,session_2026-07-17,5759,12,NaN,NaN,0.452556,0.511042,0.481799,0.117516


## 7. 모델·OOF/Test 예측·지표 저장

체크포인트에는 model state와 함께 input/target scaler, 선택 epoch, feature 이름, split 역할을 저장한다. 추후 inference는 이 scaler를 반드시 재사용해야 한다.

In [8]:
def scaler_to_serializable(scaler):
    if scaler is None:
        return None
    return {key: np.asarray(value).tolist() for key, value in scaler.items()}


def make_prediction_frame(indices, phase):
    frame = metadata.iloc[indices][[
        'sample_id', 'session', 'symbol', 'decision_timestamp', 'entry_timestamp',
        'target_tp3_3m', 'mfe_3m', 'mae_3m',
    ]].reset_index(drop=True).copy()
    frame['p_tp3_random_entry_3m'] = y_soft[indices]
    if phase == 'oof':
        frame['oof_fold'] = split.iloc[indices]['oof_fold'].to_numpy()
        hard_raw = oof_raw_predictions['open_hard'][selected_epochs['open_hard']][indices]
        soft_raw = oof_raw_predictions['random_soft'][selected_epochs['random_soft']][indices]
        reg_raw = oof_raw_predictions['mfe_mae'][selected_epochs['mfe_mae']][indices]
        reg_restored = np.full_like(reg_raw, np.nan)
        fold_numbers = split.iloc[indices]['oof_fold'].to_numpy()
        for fold in fold_document['folds']:
            fold_number = int(fold['fold'])
            mask = fold_numbers == fold_number
            reg_restored[mask] = inverse_regression_target(
                reg_raw[mask], oof_target_scalers['mfe_mae'][fold_number]
            )
    else:
        hard_raw = test_raw_predictions['open_hard']
        soft_raw = test_raw_predictions['random_soft']
        reg_raw = test_raw_predictions['mfe_mae']
        reg_restored = inverse_regression_target(reg_raw, final_target_scalers['mfe_mae'])

    frame['pred_open_hard_probability'] = clipped_probability(hard_raw.reshape(-1)).astype(np.float32)
    frame['pred_random_soft_probability'] = clipped_probability(soft_raw.reshape(-1)).astype(np.float32)
    frame['pred_mfe_3m'] = reg_restored[:, 0]
    frame['pred_downside_3m'] = reg_restored[:, 1]
    frame['pred_utility_3m'] = reg_restored[:, 0] - reg_restored[:, 1]
    return frame


oof_predictions = make_prediction_frame(oof_indices, 'oof')
test_predictions = make_prediction_frame(test_indices, 'test')

model_paths = {}
for strategy, model in final_models.items():
    path = MODEL_ROOT / f'{strategy}.pt'
    checkpoint = {
        'model_class': 'SmallTimeMixerPP',
        'implementation_scope': 'task-specific compact TimeMixer++/TimeMixer PDM adaptation',
        'strategy': strategy,
        'selected_epoch': selected_epochs[strategy],
        'model_config': {
            key: CONFIG[key]
            for key in ['input_features', 'context_features', 'd_model', 'd_ff', 'num_blocks', 'moving_average', 'dropout']
        },
        'output_dim': STRATEGIES[strategy]['output_dim'],
        'state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
        'input_scaler': scaler_to_serializable(final_input_scalers[strategy]),
        'target_scaler': scaler_to_serializable(final_target_scalers[strategy]),
        'sequence_features': schema['sequence_features'],
        'context_features': schema['context_features'],
        'data_version': DATA_VERSION,
        'train_sessions': fold_document['train_sessions'],
        'test_sessions': fold_document['test_sessions'],
        'test_is_pristine': False,
        'seed': SEED,
    }
    torch.save(checkpoint, path)
    model_paths[strategy] = path

artifact_paths = {
    'fold_metrics': RESULT_ROOT / 'oof_daily_metrics.parquet',
    'epoch_selection': RESULT_ROOT / 'oof_epoch_selection.parquet',
    'test_metrics': RESULT_ROOT / 'test_metrics.parquet',
    'training_history': RESULT_ROOT / 'training_history.parquet',
    'oof_predictions': RESULT_ROOT / 'oof_predictions.parquet',
    'test_predictions': RESULT_ROOT / 'test_predictions.parquet',
    'manifest': RESULT_ROOT / 'manifest.json',
}

fold_metrics.to_parquet(artifact_paths['fold_metrics'], index=False, compression='zstd')
epoch_selection.to_parquet(artifact_paths['epoch_selection'], index=False, compression='zstd')
test_metrics.to_parquet(artifact_paths['test_metrics'], index=False, compression='zstd')
training_history.to_parquet(artifact_paths['training_history'], index=False, compression='zstd')
oof_predictions.to_parquet(artifact_paths['oof_predictions'], index=False, compression='zstd')
test_predictions.to_parquet(artifact_paths['test_predictions'], index=False, compression='zstd')

manifest = {
    'experiment': 'timemixerpp_ohlc_60m_v1',
    'created_by_notebook': 'notebooks/05_timemixerpp_training.ipynb',
    'data_version': DATA_VERSION,
    'random_label_version': RANDOM_VERSION,
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'seed': SEED,
    'config': CONFIG,
    'selected_epochs': selected_epochs,
    'selection_data': 'Train expanding walk-forward OOF only',
    'test_role': fold_document['test_role'],
    'test_is_pristine': False,
    'model_implementation_scope': 'task-specific compact TimeMixer++/TimeMixer PDM adaptation; not byte-for-byte official benchmark reproduction',
    'model_paths': {key: str(value) for key, value in model_paths.items()},
    'result_paths': {key: str(value) for key, value in artifact_paths.items()},
}
artifact_paths['manifest'].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

artifact_table = []
for name, path in {**model_paths, **artifact_paths}.items():
    artifact_table.append({
        'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2
    })
display(pd.DataFrame(artifact_table))
print('저장 완료')

,artifact,path,size_mb
0,open_hard,/home/user/urbandatalab/YSLee/model/timemixerp...,0.536448
1,random_soft,/home/user/urbandatalab/YSLee/model/timemixerp...,0.536809
2,mfe_mae,/home/user/urbandatalab/YSLee/model/timemixerp...,0.536271
3,fold_metrics,/home/user/urbandatalab/YSLee/results/training...,0.017730
4,epoch_selection,/home/user/urbandatalab/YSLee/results/training...,0.014258
5,test_metrics,/home/user/urbandatalab/YSLee/results/training...,0.015155
6,training_history,/home/user/urbandatalab/YSLee/results/training...,0.007877
7,oof_predictions,/home/user/urbandatalab/YSLee/results/training...,0.793894
8,test_predictions,/home/user/urbandatalab/YSLee/results/training...,0.284562
9,manifest,/home/user/urbandatalab/YSLee/results/training...,0.002214


저장 완료


## 8. 결과 요약

OOF는 epoch 선택에 사용된 Train 내부 미래 날짜 예측이고, Test는 마지막 두 날짜의 개발용 진단 결과다. 두 수치를 직접 비교해 날짜 이동에 따른 일반화 하락을 확인한다. Test가 좋아도 새로운 미래 날짜로 pristine 검증하기 전에는 최종 성능으로 간주하지 않는다.

In [9]:
summary_rows = []
for strategy, spec in STRATEGIES.items():
    metric = spec['selection_metric']
    selected_epoch = selected_epochs[strategy]
    oof_row = epoch_selection[
        epoch_selection['strategy'].eq(strategy)
        & epoch_selection['epoch'].eq(selected_epoch)
    ].iloc[0]
    test_row = test_metrics[
        test_metrics['strategy'].eq(strategy)
        & test_metrics['scope'].eq('all_test')
    ].iloc[0]
    summary_rows.append({
        'strategy': strategy,
        'selected_epoch': selected_epoch,
        'primary_metric': metric,
        'OOF': float(oof_row[metric]),
        'Test': float(test_row[metric]),
        'Test_minus_OOF': float(test_row[metric] - oof_row[metric]),
    })

final_summary = pd.DataFrame(summary_rows)
display(final_summary)
print('주의: soft_bce는 낮을수록 좋고, 나머지 선택 지표는 높을수록 좋습니다.')
print('다음 단계: threshold를 아직 정하지 말고 ModernTCN을 같은 split/loss에 학습해 구조 효과부터 비교합니다.')

,strategy,selected_epoch,primary_metric,OOF,Test,Test_minus_OOF
0,open_hard,12,hard_pr_auc,0.385697,0.288774,-0.096922
1,random_soft,12,soft_bce,0.301507,0.237546,-0.063961
2,mfe_mae,12,mean_target_spearman,0.498330,0.504952,0.006622


주의: soft_bce는 낮을수록 좋고, 나머지 선택 지표는 높을수록 좋습니다.
다음 단계: threshold를 아직 정하지 말고 ModernTCN을 같은 split/loss에 학습해 구조 효과부터 비교합니다.
